# Plot maps of biases

In [25]:
import os
import xarray as xr
import pandas as pd
import geopandas as gpd
import plotnine as p9
import mizani
from plotnine import (
    ggplot, aes, 
    geom_point, geom_raster, geom_tile,
    facet_wrap, facet_grid,
    coord_equal,
    xlim, ylim, ggtitle,
    scale_fill_distiller, scale_fill_gradient2,
    theme, element_blank, element_text
)
import regionmask

## settings and data-prep

In [26]:
ref_data = "cerra"
variable = "pr"
path_fig = os.path.abspath(os.path.join(os.getcwd(), "..", "fig/maps-modelmeans"))
os.makedirs(path_fig, exist_ok=True)

In [27]:
gpd_regions = gpd.read_file("../data/regions.gpkg")
regions = regionmask.Regions.from_geodataframe(gpd_regions)

In [28]:
xds = xr.open_dataset("../intermediate-results/" + ref_data + "_" + variable + "_CMIP6_1991-2020_spatial_bias.nc")

In [29]:
mask = regions.mask_3D(xds["lon"], xds["lat"], drop=False)
df_mask = mask.to_dataframe().reset_index()
df_mask_tosubset = df_mask[["region", "rlat", "rlon", "names", "mask"]][df_mask["mask"]]

In [30]:
df = xds.to_dataframe().reset_index()
df_regions = pd.merge(df, df_mask_tosubset).dropna()

In [31]:
df_regions_mm = df_regions.groupby(["season","rlat","rlon","region","names"])[[variable]].mean()
df_regions_mm = df_regions_mm.reset_index()

In [32]:
# df1 = df_regions[(df_regions["dset_id"]=="CNRM-ALADIN64E1") & 
#     (df_regions["season"] == "DJF")]

In [33]:
# (
#     ggplot(df1[df1["region"]==0], aes("rlon","rlat",fill="tas"))
#     + geom_raster()
#     # + scale_fill_distiller(type="div", limits=[-5,5], direction=-1)
#     + scale_fill_gradient2(mid="#f5f5f5", high="#d73027", low="#4575b4")
#     # + scale_fill_gradient2()
#     + facet_wrap("season")
#     + theme_bw()
#     + ggtitle(regions[0].name)
# )

In [34]:
# gg = (
#     ggplot(df_tas_regions[(df_regions["region"]==0) & 
#                           (df_regions["season"]=="DJF")], 
#            aes("rlon","rlat",fill=variable))
#     + geom_raster()
#     + scale_fill_gradient2(name = variable + "\n bias \n E-OBS", 
#                            mid="#f5f5f5", high="#d73027", low="#4575b4")
#     # + facet_grid("season", "dset_id")
#     + facet_wrap("dset_id")
#     + coord_equal()
#     + theme_bw()
#     + theme(axis_title=element_blank(), 
#             axis_ticks=element_blank(), 
#             axis_text=element_blank())
#     + ggtitle(regions[0].name)
# )
# gg.show()
# # gg.save("test.png", width=12, height=8, dpi=300)

## Plots

In [35]:
regions

<regionmask.Regions 'unnamed'>
overlap:  None

Regions:
0 Alp        Alps
1 Car Carpathians
2 Pyr    Pyrenees
3 Sca     Scandes

[4 regions]

In [36]:
if variable == "pr":
    col_high = "#01665e"
    col_low = "#8c510a"
else:
    col_low = "#4575b4"
    col_high = "#d73027"

col_limits = [df_regions_mm[variable].min(), df_regions_mm[variable].max()]
if col_limits[1] > 200:
    col_limits[1] = 200.0

if ref_data == "eobs":
    lbl_ref = "E-OBS"
elif ref_data == "cerra":
    lbl_ref = "CERRA"

def f_plot(region):
    gg = (
        ggplot(df_regions_mm[df_regions_mm["names"] == region], aes("rlon","rlat",fill=variable))
        + geom_raster()
        + facet_wrap(" ~ season")
        # + facet_grid("names ~ season")
        + coord_equal()
        + p9.theme_classic(8)
        + theme(axis_title=element_blank(), 
                axis_ticks=element_blank(), 
                axis_text=element_blank(),
                legend_position="none")
        + ggtitle(region)
        # + ggtitle(f"{variable} - {season} - {regions[i].name}")
    )
    if variable == "pr":
        gg = gg + (
            scale_fill_gradient2(name=variable + "\n bias \n " + lbl_ref + " \n [%]", 
                                 mid="#f5f5f5", high=col_high, low=col_low,
                                 limits=col_limits, oob=mizani.bounds.squish)
        )
    else:
        gg = gg + (
            scale_fill_gradient2(name=variable + "\n bias \n " + lbl_ref + " \n [°C]", 
                                 limits=col_limits, mid="#f5f5f5", high=col_high, low=col_low)
        )
    return(gg)

gg0 = f_plot("Alps")
gg1 = f_plot("Carpathians")
gg2 = f_plot("Pyrenees")
gg3 = f_plot("Scandes")+theme(legend_position="right")

gg_out = (gg0 | gg1) / (gg2 | gg3)
# gg_out = gg0 | (gg1 / gg2) - gg3
# gg_out = (gg0 / gg1 / gg2) | gg3
filepath_save = f"{path_fig}/{ref_data}_{variable}.png"
gg_out.save(filepath_save, width=16, height=8, dpi=300, verbose=False)